In [0]:
from pyspark.sql import functions as F

billing_bronze = spark.table(
    "healthcare.default.bronze_billing"
)

patients_bronze = spark.table(
    "healthcare.default.bronze_patients"
)

treatments_bronze = spark.table(
    "healthcare.default.bronze_treatments"
)

print("Billing:", billing_bronze.count())
print("Patients:", patients_bronze.count())
print("Treatments:", treatments_bronze.count())

Billing: 200
Patients: 50
Treatments: 200


In [0]:
display(billing_bronze)

bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
B001,P034,T001,2023-08-09,3941.97,Insurance,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,64c267b37c111ac325693a2aa5f168aeedfe141d6f8ec637e61bd517a302ab81,BRONZE
B002,P032,T002,2023-06-09,4158.44,Insurance,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,51aef527977c21c78246db1424579f9a267fc70c0812baca15fe7b7c801bba88,BRONZE
B003,P048,T003,2023-06-28,3731.55,Insurance,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,5687b56253176f7659c7e2d1926261a7b99ea31a428ef9070bdd925fd5403377,BRONZE
B004,P025,T004,2023-09-01,4799.86,Insurance,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,96c08dd3636e9b7937b55de8e0e0a02d87930a67fdf6a021272420325572a8c2,BRONZE
B005,P040,T005,2023-07-06,582.05,Credit Card,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,a6da7bf8fc64c43f3f413187487c45b16da740a08f6ba844266c8c30f8ae0b3a,BRONZE
B006,P045,T006,2023-06-19,1381.0,Insurance,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,4b58cebafeb12c2028525e00ad7733d1653bcddd8aa76db0844403c202f24210,BRONZE
B007,P001,T007,2023-04-09,534.03,Cash,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,7b61c143e2808abc634f1655ddface9900fc7538f62d83efedc7b4b32b81a335,BRONZE
B008,P016,T008,2023-05-24,3413.64,Cash,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,c45b3fe2dc63b386f85afbff54c8ae741da5b829ee31b3b3d626c8e3f2ad3364,BRONZE
B009,P039,T009,2023-03-05,4541.14,Credit Card,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,1ef85b679800862fcbbecef5d034b91374f3664cfe09782ee0fcae45adda2245,BRONZE
B010,P005,T010,2023-01-13,1595.67,Cash,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,878b04300ef828e40c6d1ebbc603912e7c59e4987d2c58a9bfb7eba18021b911,BRONZE


In [0]:
duplicate_bills = (
    billing_bronze
    .groupBy("bill_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate bill IDs:",
    duplicate_bills.count()
)

display(duplicate_bills)

Duplicate bill IDs: 0


bill_id,count


In [0]:
billing_nulls = billing_bronze.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in billing_bronze.columns
])

display(billing_nulls)

bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
display(
    billing_bronze.select(
        F.min("amount").alias("minimum_amount"),
        F.max("amount").alias("maximum_amount"),
        F.avg("amount").alias("average_amount"),
        F.sum("amount").alias("total_amount")
    )
)

minimum_amount,maximum_amount,average_amount,total_amount
534.03,4973.63,2756.2492500000003,551249.8500000001


In [0]:
invalid_amounts = (
    billing_bronze
    .filter(
        F.col("amount").isNull()
        | (F.col("amount") <= 0)
    )
)

print(
    "Bills with invalid amount:",
    invalid_amounts.count()
)

display(invalid_amounts)

Bills with invalid amount: 0


bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
display(
    billing_bronze
    .groupBy("payment_method")
    .count()
    .orderBy("payment_method")
)

payment_method,count
Cash,61
Credit Card,75
Insurance,64


In [0]:
display(
    billing_bronze
    .groupBy("payment_status")
    .count()
    .orderBy("payment_status")
)

payment_status,count
Failed,67
Paid,64
Pending,69


In [0]:
invalid_billing_patients = (
    billing_bronze
    .join(
        patients_bronze
        .select("patient_id")
        .distinct(),
        on="patient_id",
        how="left_anti"
    )
)

print(
    "Bills with invalid patient_id:",
    invalid_billing_patients.count()
)

display(invalid_billing_patients)

Bills with invalid patient_id: 0


patient_id,bill_id,treatment_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
invalid_billing_treatments = (
    billing_bronze
    .join(
        treatments_bronze
        .select("treatment_id")
        .distinct(),
        on="treatment_id",
        how="left_anti"
    )
)

print(
    "Bills with invalid treatment_id:",
    invalid_billing_treatments.count()
)

display(invalid_billing_treatments)

Bills with invalid treatment_id: 0


treatment_id,bill_id,patient_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer


In [0]:
display(
    billing_bronze.select(
        F.min("bill_date").alias("earliest_bill_date"),
        F.max("bill_date").alias("latest_bill_date")
    )
)

earliest_bill_date,latest_bill_date
2023-01-01,2023-12-30


In [0]:
print("Total bills:", billing_bronze.count())
print("Duplicate bill IDs:", duplicate_bills.count())
print("Invalid billing amounts:", invalid_amounts.count())
print("Invalid patient IDs:", invalid_billing_patients.count())
print("Invalid treatment IDs:", invalid_billing_treatments.count())

Total bills: 200
Duplicate bill IDs: 0
Invalid billing amounts: 0
Invalid patient IDs: 0
Invalid treatment IDs: 0


In [0]:
silver_billing = (
    billing_bronze
    .filter(
        F.col("bill_id").isNotNull()
        & F.col("patient_id").isNotNull()
        & F.col("treatment_id").isNotNull()
        & F.col("bill_date").isNotNull()
        & F.col("amount").isNotNull()
        & (F.col("amount") > 0)
        & F.col("payment_method").isNotNull()
        & F.col("payment_status").isNotNull()
    )
    .dropDuplicates(["bill_id"])
    .withColumn(
        "payment_method",
        F.initcap(F.trim(F.col("payment_method")))
    )
    .withColumn(
        "payment_status",
        F.initcap(F.trim(F.col("payment_status")))
    )
    .withColumn(
        "amount",
        F.round(F.col("amount"), 2)
    )
)

print("Silver billing rows:", silver_billing.count())

display(silver_billing)

Silver billing rows: 200


bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
B001,P034,T001,2023-08-09,3941.97,Insurance,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,64c267b37c111ac325693a2aa5f168aeedfe141d6f8ec637e61bd517a302ab81,BRONZE
B002,P032,T002,2023-06-09,4158.44,Insurance,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,51aef527977c21c78246db1424579f9a267fc70c0812baca15fe7b7c801bba88,BRONZE
B003,P048,T003,2023-06-28,3731.55,Insurance,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,5687b56253176f7659c7e2d1926261a7b99ea31a428ef9070bdd925fd5403377,BRONZE
B004,P025,T004,2023-09-01,4799.86,Insurance,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,96c08dd3636e9b7937b55de8e0e0a02d87930a67fdf6a021272420325572a8c2,BRONZE
B005,P040,T005,2023-07-06,582.05,Credit Card,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,a6da7bf8fc64c43f3f413187487c45b16da740a08f6ba844266c8c30f8ae0b3a,BRONZE
B006,P045,T006,2023-06-19,1381.0,Insurance,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,4b58cebafeb12c2028525e00ad7733d1653bcddd8aa76db0844403c202f24210,BRONZE
B007,P001,T007,2023-04-09,534.03,Cash,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,7b61c143e2808abc634f1655ddface9900fc7538f62d83efedc7b4b32b81a335,BRONZE
B008,P016,T008,2023-05-24,3413.64,Cash,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,c45b3fe2dc63b386f85afbff54c8ae741da5b829ee31b3b3d626c8e3f2ad3364,BRONZE
B009,P039,T009,2023-03-05,4541.14,Credit Card,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,1ef85b679800862fcbbecef5d034b91374f3664cfe09782ee0fcae45adda2245,BRONZE
B010,P005,T010,2023-01-13,1595.67,Cash,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,878b04300ef828e40c6d1ebbc603912e7c59e4987d2c58a9bfb7eba18021b911,BRONZE


In [0]:
silver_billing.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.silver_billing"
    )

print("silver_billing created successfully")

silver_billing created successfully


In [0]:
silver_billing_check = spark.table(
    "healthcare.default.silver_billing"
)

print(
    "Silver billing count:",
    silver_billing_check.count()
)

display(silver_billing_check)

Silver billing count: 200


bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
B046,P019,T046,2023-12-20,1526.36,Cash,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,0f6e49a76e3e98f453816f4cf64dee5a935eb21f9d4e585f8a2cd656becd14b3,BRONZE
B073,P040,T073,2023-12-24,2259.08,Credit Card,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,90a11674c159be581e007b083922aa9898781d0918bbddca04f61e0c3bd5889f,BRONZE
B100,P029,T100,2023-03-02,1551.7,Credit Card,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,67dbc88471618097565bf23f6e8ec0db0d11c9c451a0f9dd8652154a6fe43dd1,BRONZE
B102,P025,T102,2023-10-25,4460.36,Credit Card,Pending,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,f89a5283e3f7d241b782c4d0f0beaf7602e409e8123ba455c0a490be9b6c07ae,BRONZE
B150,P047,T150,2023-08-16,2286.42,Credit Card,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,ca4c334285e398533aaf9ce0e7dc79caa8d77721547bc102ecd416d9b202ac0d,BRONZE
B165,P031,T165,2023-04-04,4126.66,Cash,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,55eec32aacec208ad3c940689a3a904fd644480e5c2460c56c55be9af04ad00c,BRONZE
B168,P023,T168,2023-09-29,864.14,Credit Card,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,6f4414c54030add5326967c5cd2aadf66e7d4ca92d002d5f32d48a31db788de5,BRONZE
B198,P022,T198,2023-05-15,3383.72,Cash,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,a732ca4944000dcb258df1e3ba9bdebb5dc3537ff436e5959f5b45aa82b737f9,BRONZE
B199,P017,T199,2023-05-01,1472.17,Credit Card,Paid,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,94620e6210322987718fd263eae7da4a4733201992f2ee68a9efdb672f746691,BRONZE
B021,P028,T021,2023-04-24,2926.23,Insurance,Failed,BATCH_20260810_144925_920fea,SRC003,billing,billing.csv,2026-08-10T14:49:35.795Z,2026-08-10,b0329bb1a9818740c2f08f6a1b3407175fd0558bed44b3bd561df49da64abfa5,BRONZE


In [0]:
print("===== SILVER TABLE VALIDATION =====")

silver_patients = spark.table(
    "healthcare.default.silver_patients"
)

silver_appointments = spark.table(
    "healthcare.default.silver_appointments"
)

silver_doctors = spark.table(
    "healthcare.default.silver_doctors"
)

silver_treatments = spark.table(
    "healthcare.default.silver_treatments"
)

silver_billing = spark.table(
    "healthcare.default.silver_billing"
)

print("Patients:", silver_patients.count())
print("Appointments:", silver_appointments.count())
print("Doctors:", silver_doctors.count())
print("Treatments:", silver_treatments.count())
print("Billing:", silver_billing.count())

===== SILVER TABLE VALIDATION =====
Patients: 50
Appointments: 200
Doctors: 10
Treatments: 200
Billing: 200


In [0]:
from pyspark.sql import functions as F

patients = spark.table(
    "healthcare.default.silver_patients"
)

appointments = spark.table(
    "healthcare.default.silver_appointments"
)

treatments = spark.table(
    "healthcare.default.silver_treatments"
)

billing = spark.table(
    "healthcare.default.silver_billing"
)

print("Patients:", patients.count())
print("Appointments:", appointments.count())
print("Treatments:", treatments.count())
print("Billing:", billing.count())

Patients: 50
Appointments: 200
Treatments: 200
Billing: 200


In [0]:
patient_appointments = (
    appointments
    .groupBy("patient_id")
    .agg(
        F.count("appointment_id").alias("total_appointments"),
        F.sum(
            F.when(F.col("status") == "Completed", 1).otherwise(0)
        ).alias("completed_appointments"),
        F.sum(
            F.when(F.col("status") == "Cancelled", 1).otherwise(0)
        ).alias("cancelled_appointments"),
        F.sum(
            F.when(F.col("status") == "No-show", 1).otherwise(0)
        ).alias("no_show_appointments"),
        F.min("appointment_date").alias("first_appointment_date"),
        F.max("appointment_date").alias("last_appointment_date")
    )
)

display(patient_appointments.orderBy("patient_id"))

patient_id,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date
P001,4,0,1,1,2023-01-16,2023-04-09
P002,3,0,1,0,2023-01-20,2023-10-06
P003,2,0,0,0,2023-08-16,2023-08-26
P004,2,2,0,0,2023-02-04,2023-07-07
P005,8,2,0,4,2023-01-01,2023-11-14
P007,4,0,3,0,2023-01-07,2023-12-30
P008,2,1,0,0,2023-04-06,2023-05-02
P009,4,1,1,1,2023-03-21,2023-10-22
P010,6,1,1,3,2023-03-27,2023-09-28
P011,2,1,1,0,2023-04-18,2023-07-04


In [0]:
patient_treatments = (
    treatments
    .groupBy("appointment_id")
    .agg(
        F.count("treatment_id").alias("total_treatments"),
        F.sum("cost").alias("total_treatment_cost"),
        F.avg("cost").alias("average_treatment_cost"),
        F.min("treatment_date").alias("first_treatment_date"),
        F.max("treatment_date").alias("last_treatment_date")
    )
)

display(
    patient_treatments
    .orderBy("appointment_id")
)

appointment_id,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
A001,1,3941.97,3941.97,2023-08-09,2023-08-09
A002,1,4158.44,4158.44,2023-06-09,2023-06-09
A003,1,3731.55,3731.55,2023-06-28,2023-06-28
A004,1,4799.86,4799.86,2023-09-01,2023-09-01
A005,1,582.05,582.05,2023-07-06,2023-07-06
A006,1,1381.0,1381.0,2023-06-19,2023-06-19
A007,1,534.03,534.03,2023-04-09,2023-04-09
A008,1,3413.64,3413.64,2023-05-24,2023-05-24
A009,1,4541.14,4541.14,2023-03-05,2023-03-05
A010,1,1595.67,1595.67,2023-01-13,2023-01-13
